# OQIM AI - Shahar Trafikini Boshqarish Tizimi

Ushbu loyiha shahar chorrahalarida sun'iy intellekt (Reinforcement Learning va Machine Learning) yordamida tirbandliklarni aniqlash va svetoforlarni optimal boshqarishni amalga oshiradi. 

Loyiha taraqqiyoti uch bosqichda tahlil qilinadi:
1. **Hozirgi holat** - Qat'iy vaqtga asoslangan samarasiz tizim.
2. **AI joriy etilgandan keyingi holat** - Q-Learning orqali o'z-o'zini moslashtiruvchi aqlli svetaforlar.
3. **Kelajak holati (V2X & Yashil To'lqin)** - Avtomobillar va svetaforlar o'rtasidagi to'liq integratsiya, uzluksiz harakat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import random

print("✅ Loyiha kutubxonalari yuklandi!")

## 1-Qism: Tizim Evolyutsiyasini Taqqoslash (Matematik Isbot)

Ushbu qismda bir xil transport oqimi sharoitida chorraha 3 xil stsenariyda qanday ishlashi simulyatsiya qilinadi:

1. **Hozirgi holat (Fiks vaqt):** Svetafor doim 20 sekund yashil beradi. Oqim ko'payganda tirbandlik qattiq o'sadi.
2. **AI ishlagandan keyingi holat (Q-Learning):** AI mashinalar soniga qarab yashil chiroq vaqtini (masalan, 30-40 sekundgacha) dinamik o'zgartiradi va isrofni oldini oladi.
3. **Kelajak holat (AI + V2X Yashil To'lqin):** Avtomobillar svetaforga yetib kelmasdan avval AI bilan ma'lumot almashadi (Vehicle-to-Everything). Mashinalar o'z tezligini yashil chiroqqa tushadigan qilib moslaydi, natijada chorrahada umuman to'xtalish bo'lmaydi.

In [ ]:
class TrafficEnvironment:
    def __init__(self):
        self.queue = 0 
    
    def step(self, green_time, incoming_cars):
        self.queue += incoming_cars
        cars_passed = green_time / 2.0 
        self.queue = max(0, self.queue - cars_passed)
        return self.queue

# 1. Sun'iy Intellektni O'qitish (Q-Learning)
actions = [10, 20, 30, 40, 50, 60] 
q_table = {}

def get_state(queue):
    return min(int(queue // 5), 15) 

for episode in range(5000):
    queue = random.randint(0, 50)
    state = get_state(queue)
    if state not in q_table: 
        q_table[state] = {a: 0.0 for a in actions}
    
    action = random.choice(actions)
    cars_passed = action / 2.0
    remaining = max(0, queue - cars_passed)
    
    penalty_queue = -remaining * 2 
    wasted_time = max(0, action - (queue * 2))
    penalty_wasted = -wasted_time * 0.5 
    
    reward = penalty_queue + penalty_wasted
    
    q_table[state][action] = q_table[state][action] + 0.2 * (reward - q_table[state][action])

# 2. UCH BOSQICHLI TAQQOSLASH (Hozirgi vs AI vs Kelajak)
np.random.seed(101)
incoming_traffic = np.random.poisson(16, 100) 

env_current = TrafficEnvironment()
env_ai = TrafficEnvironment()
env_future = TrafficEnvironment()

history_current = []
history_ai = []
history_future = []

for cars in incoming_traffic:
    # 1. Hozirgi holat (Fiks 20 sek)
    q_curr = env_current.step(20, cars)
    history_current.append(q_curr)
    
    # 2. AI ishlagandan keyingi holat
    state_ai = get_state(env_ai.queue + cars)
    if state_ai in q_table:
        best_action = max(q_table[state_ai], key=q_table[state_ai].get)
    else:
        best_action = 30
    q_ai = env_ai.step(best_action, cars)
    history_ai.append(q_ai)
    
    # 3. Kelajak holat (V2X + Yashil to'lqin)
    # Avtomobillar kelishdan oldin tezligini moslaydi, shuning uchun kutish nolga yaqinlashadi
    q_fut = env_future.step(50, cars * 0.2) # Mashinalarning 80% to'xtamasdan o'tib ketadi
    history_future.append(q_fut)

# Natijalarni grafikda ko'rsatish
plt.figure(figsize=(12, 6))
plt.plot(history_current, label='1. Hozirgi holat (Qat\'iy vaqtli svetafor)', color='red', alpha=0.8, linewidth=2.5)
plt.plot(history_ai, label='2. AI joriy etilgandan so\'ng (Q-Learning)', color='blue', alpha=0.9, linewidth=2.5)
plt.plot(history_future, label='3. Kelajak holati (V2X va Yashil To\'lqin)', color='green', alpha=0.9, linewidth=2.5, linestyle='--')

plt.title('Chorrahada Navbatda Turgan Mashinalar Soni (Tirbandlik Evolyutsiyasi)')
plt.xlabel('Vaqt (Tsikllar)')
plt.ylabel('Kutayotgan Mashinalar Soni')
plt.legend(loc='upper left')
plt.grid(True)
plt.fill_between(range(len(history_current)), history_current, history_ai, color='red', alpha=0.1)
plt.fill_between(range(len(history_ai)), history_ai, history_future, color='blue', alpha=0.1)
plt.show()

print("Tahlil Natijalari:")
print(f"🔴 1. Hozirgi holatda o'rtacha kutish: {np.mean(history_current):.1f} ta mashina (Xavotirli daraja)")
print(f"🔵 2. AI ishlagandan so'ng o'rtacha kutish: {np.mean(history_ai):.1f} ta mashina (Optimal daraja)")
print(f"🟢 3. Kelajak (V2X) tizimida o'rtacha kutish: {np.mean(history_future):.1f} ta mashina (Mukammal daraja)")

if np.mean(history_current) > 0:
    improvement_ai = 100 - (np.mean(history_ai) / np.mean(history_current) * 100)
    improvement_future = 100 - (np.mean(history_future) / np.mean(history_current) * 100)
    print(f"\n🚀 Loyiha samaradorligi: Tirbandlik AI orqali {improvement_ai:.1f}% ga, Kelajakda esa {improvement_future:.1f}% ga kamaytirildi.")

## 2-Qism: DBSCAN yordamida Tirbandliklarni Klasterlash (Density-based Clustering)

Haqiqiy tirbandliklarni aniqlashda standart masofa algoritmlari (masalan, K-Means) o'rniga zichlikka qarab ishlaydigan **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** algoritmidan foydalanildi.

Bu algoritm mashinalar bir-biriga juda tiqilib qolgan (zich) joylarni avtomatik qidirib topadi va oraliq masofadagi yakka mashinalarni chalg'ituvchi "shovqin" (noise) sifatida e'tiborsiz qoldiradi. Bu haqiqiy probkalarni lokalizatsiya qilishda eng aniq yechim hisoblanadi.

In [ ]:
# Tasodifiy 300 ta mashina lokatsiyasini yaratish
np.random.seed(0)
cluster_1 = np.random.normal([30, 30], 5, (100, 2))
cluster_2 = np.random.normal([70, 70], 4, (100, 2))
noise = np.random.uniform(0, 100, (100, 2)) 
vehicles = np.vstack((cluster_1, cluster_2, noise))

dbscan = DBSCAN(eps=6, min_samples=8)
labels = dbscan.fit_predict(vehicles)

plt.figure(figsize=(8, 6))

plt.scatter(vehicles[labels == -1, 0], vehicles[labels == -1, 1], c='gray', label='Erkin Harakat (Shovqin)', alpha=0.5, s=20)

unique_labels = set(labels)
unique_labels.discard(-1)

colors = ['red', 'purple', 'orange', 'brown']
for i, label in enumerate(unique_labels):
    plt.scatter(vehicles[labels == label, 0], vehicles[labels == label, 1], c=colors[i % len(colors)], label=f'Tirbandlik Zonasi {label+1}', s=60)

plt.title("DBSCAN: Zichlikka Asoslangan Tirbandlik Tahlili")
plt.legend()
plt.grid(True)
plt.show()

print("Izoh: DBSCAN fiks klasterlarga emas, haqiqiy zichlik markazlariga e'tibor qaratdi.")